# 平面图分析 Pipeline 原型

**对应论文**：*Automatic Reconstruction of Semantic 3D Models from 2D Floor Plans*（MVA 2023）


**第一步**：墙体分割（FPN+ResNet）+ 门窗检测（Faster-RCNN）


---

## 流程

```

原始平面图

  → 滑动窗口切片（Section 2.1）

  → 墙体分割 FPN+ResNet（Section 2.3）

  → 门窗检测 Faster-RCNN（Section 2.2）

  → 拼接 + 可视化

```

## 0. 安装依赖

In [ ]:
# 如果环境中尚未安装，取消注释执行

# !pip install torch torchvision opencv-python matplotlib

## 1. 导入库 & 全局配置

In [ ]:
import time

import logging

from pathlib import Path



import cv2

import numpy as np

import matplotlib.pyplot as plt

import matplotlib.patches as mpatches

import torch

import torch.nn as nn

import torch.nn.functional as F

from torchvision.models import resnet50, ResNet50_Weights

from torchvision.ops import FeaturePyramidNetwork

from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchvision import transforms



logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

logger = logging.getLogger(__name__)



%matplotlib inline

plt.rcParams['figure.dpi'] = 120

print('导入完成')

In [ ]:
# ══ 全局配置（先写死，跑通后抽成 config.py）══



TILE_SIZE    = 512      # 滑动窗口大小（论文 Section 2.1）

TILE_OVERLAP = 64       # 重叠像素，消除拼接边缘

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'



WALL_CLASSES        = ['background', 'wall']

SYMBOL_CLASSES      = ['background', 'door', 'window']

WALL_THRESHOLD      = 0.5    # 墙体概率阈值

SYMBOL_SCORE_THRESH = 0.5    # 检测框置信度

NMS_THRESH          = 0.3



# ImageNet 归一化（两个模型都用）

NORM_MEAN = [0.485, 0.456, 0.406]

NORM_STD  = [0.229, 0.224, 0.225]



# 输入图片路径 ← 改成你的路径

IMAGE_PATH  = 'company_test/floorplan1.png'

OUTPUT_DIR  = 'outputs'

WALL_CKPT   = None   # 有训练好的权重时填路径，如 'checkpoints/wall_seg.pth'

SYMBOL_CKPT = None



print(f'device: {DEVICE}')

print(f'image : {IMAGE_PATH}')

## 2. 预处理 & 滑动窗口

**论文 Section 2.1**：滑动窗口处理不同尺寸图像，边界用反射 padding 补足

In [ ]:
def load_image(path: str) -> np.ndarray:

    """加载平面图，统一转为 RGB uint8"""

    img = cv2.imread(path)

    if img is None:

        raise FileNotFoundError(f'无法读取图像：{path}')

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    logger.info('图像加载成功，尺寸：%s', img.shape[:2])

    return img





def image_to_tensor(image: np.ndarray) -> torch.Tensor:

    """

    np.ndarray [H,W,3] uint8

    → torch.Tensor [1,3,H,W] float32（ImageNet 归一化）

    """

    t = transforms.ToTensor()(image)                     # [3,H,W] [0,1]

    t = transforms.Normalize(NORM_MEAN, NORM_STD)(t)    # ImageNet 归一化

    return t.unsqueeze(0)                                 # [1,3,H,W]





def extract_tiles(image: np.ndarray):

    """

    滑动窗口切片生成器（论文 Section 2.1）

    yield: (tile_img [TILE_SIZE,TILE_SIZE,3], x, y)

    边界用反射 padding 补足，避免黑边影响预测

    """

    h, w   = image.shape[:2]

    stride = TILE_SIZE - TILE_OVERLAP

    y = 0

    while y < h:

        x = 0

        while x < w:

            tile = image[y: y + TILE_SIZE, x: x + TILE_SIZE]

            th, tw = tile.shape[:2]

            if th < TILE_SIZE or tw < TILE_SIZE:

                tile = cv2.copyMakeBorder(

                    tile, 0, TILE_SIZE - th, 0, TILE_SIZE - tw,

                    cv2.BORDER_REFLECT_101

                )

            yield tile, x, y

            x += stride

            if x < w <= x + TILE_SIZE:

                x = max(w - TILE_SIZE, 0)

            elif x >= w:

                break

        y += stride

        if y < h <= y + TILE_SIZE:

            y = max(h - TILE_SIZE, 0)

        elif y >= h:

            break





def stitch_wall_predictions(tile_preds, orig_h, orig_w) -> np.ndarray:

    """

    将各 tile 的墙体概率图拼接回原始尺寸

    重叠区域用高斯权重加权平均，消除拼接线

    """

    accum   = np.zeros((orig_h, orig_w), dtype=np.float32)

    weights = np.zeros((orig_h, orig_w), dtype=np.float32)



    sigma = TILE_SIZE * 0.3

    cy, cx = TILE_SIZE / 2, TILE_SIZE / 2

    gy = np.exp(-0.5 * ((np.arange(TILE_SIZE) - cy) / sigma) ** 2)

    gx = np.exp(-0.5 * ((np.arange(TILE_SIZE) - cx) / sigma) ** 2)

    gauss = np.outer(gy, gx).astype(np.float32)



    for (prob, x, y) in tile_preds:

        y_end = min(y + TILE_SIZE, orig_h)

        x_end = min(x + TILE_SIZE, orig_w)

        ch, cw = y_end - y, x_end - x

        accum[y:y_end, x:x_end]   += prob[:ch, :cw] * gauss[:ch, :cw]

        weights[y:y_end, x:x_end] += gauss[:ch, :cw]



    return accum / np.maximum(weights, 1e-8)





# ── 验证：加载图像并显示 tile 切分情况 ──

image   = load_image(IMAGE_PATH)

orig_h, orig_w = image.shape[:2]

tiles   = list(extract_tiles(image))



print(f'原图尺寸：{orig_h} × {orig_w}')

print(f'切分为 {len(tiles)} 个 tile（大小 {TILE_SIZE}×{TILE_SIZE}，重叠 {TILE_OVERLAP}px）')



# 可视化 tile 边界

vis = image.copy()

for _, tx, ty in tiles:

    cv2.rectangle(vis, (tx, ty),

                  (min(tx+TILE_SIZE, orig_w-1), min(ty+TILE_SIZE, orig_h-1)),

                  (255, 0, 0), 2)

plt.figure(figsize=(10, 8))

plt.imshow(vis)

plt.title(f'滑动窗口切片（共 {len(tiles)} 个）', fontsize=14)

plt.axis('off')

plt.tight_layout()

plt.show()

## 3. 墙体分割模型

**论文 Section 2.3**：FPN + ResNet 语义分割

- Backbone：ResNet50（ImageNet 预训练）

- Neck：FPN（自顶向下特征融合）

- Head：Conv + BN + ReLU + 1×1 输出 logits

- Loss：BCE + Affinity Field Loss（Ke et al. ECCV 2018）

In [ ]:
class WallSegModel(nn.Module):

    """

    FPN + ResNet50 墙体二分类分割

    输出：[B, 2, H, W] logits（background / wall）

    """

    def __init__(self, num_classes=2):

        super().__init__()

        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        self.layer0 = nn.Sequential(

            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool

        )

        self.layer1 = backbone.layer1   # stride 4,   256ch

        self.layer2 = backbone.layer2   # stride 8,   512ch

        self.layer3 = backbone.layer3   # stride 16, 1024ch

        self.layer4 = backbone.layer4   # stride 32, 2048ch



        self.fpn = FeaturePyramidNetwork(

            in_channels_list=[256, 512, 1024, 2048],

            out_channels=256,

        )

        self.head = nn.Sequential(

            nn.Conv2d(256, 128, 3, padding=1, bias=False),

            nn.BatchNorm2d(128),

            nn.ReLU(inplace=True),

            nn.Conv2d(128, num_classes, 1),

        )



    def forward(self, x):

        size = x.shape[-2:]

        x  = self.layer0(x)

        c2 = self.layer1(x)

        c3 = self.layer2(c2)

        c4 = self.layer3(c3)

        c5 = self.layer4(c4)

        fpn_out = self.fpn({'0': c2, '1': c3, '2': c4, '3': c5})

        ref = fpn_out['0'].shape[-2:]

        fused = fpn_out['0']

        for k in ['1', '2', '3']:

            fused = fused + F.interpolate(

                fpn_out[k], size=ref, mode='bilinear', align_corners=False

            )

        logits = self.head(fused)

        return F.interpolate(logits, size=size, mode='bilinear', align_corners=False)





# ── 初始化并打印参数量 ──

wall_model = WallSegModel(num_classes=2).to(DEVICE)



if WALL_CKPT:

    ckpt  = torch.load(WALL_CKPT, map_location='cpu')

    state = ckpt.get('model_state_dict', ckpt)

    wall_model.load_state_dict({k.replace('module.', ''): v for k, v in state.items()})

    print(f'✓ 墙体分割权重已加载：{WALL_CKPT}')

else:

    print('⚠ 使用随机权重（仅验证流程），请训练后传入 WALL_CKPT')



params = sum(p.numel() for p in wall_model.parameters()) / 1e6

print(f'模型参数量：{params:.1f}M')

### 3.1 墙体分割推理

In [ ]:
def run_wall_segmentation(model, image):

    """滑动窗口墙体分割，返回概率图和二值 mask"""

    model.eval()

    orig_h, orig_w = image.shape[:2]

    tile_preds = []



    with torch.no_grad():

        for tile_img, tx, ty in extract_tiles(image):

            tensor  = image_to_tensor(tile_img).to(DEVICE)

            logits  = model(tensor)                         # [1,2,H,W]

            prob    = torch.softmax(logits, dim=1)[0, 1]   # 墙体概率

            tile_preds.append((prob.cpu().numpy(), tx, ty))



    wall_prob = stitch_wall_predictions(tile_preds, orig_h, orig_w)

    wall_mask = wall_prob >= WALL_THRESHOLD

    logger.info('墙体分割完成，墙体像素：%d', wall_mask.sum())

    return wall_prob, wall_mask





t0 = time.perf_counter()

wall_prob, wall_mask = run_wall_segmentation(wall_model, image)

print(f'耗时：{time.perf_counter()-t0:.2f}s')

print(f'墙体像素：{wall_mask.sum()}  /  总像素：{orig_h*orig_w}')



# 可视化

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image);             axes[0].set_title('原始平面图');     axes[0].axis('off')

axes[1].imshow(wall_prob, cmap='hot', vmin=0, vmax=1)

axes[1].set_title('墙体概率图');   axes[1].axis('off')

axes[2].imshow(wall_mask, cmap='gray')

axes[2].set_title('墙体二值 Mask'); axes[2].axis('off')

plt.tight_layout()

plt.show()

## 4. 符号检测模型

**论文 Section 2.2**：Faster-RCNN + ResNet50-FPN

> "门窗符号是离散、重复的图案，适合用矩形框表示"（论文原文）

In [ ]:
def build_symbol_model(num_classes=3):

    """

    Faster-RCNN + ResNet50-FPN

    类别：[background, door, window]

    """

    model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    model.roi_heads.score_thresh  = SYMBOL_SCORE_THRESH

    model.roi_heads.nms_thresh    = NMS_THRESH

    return model





symbol_model = build_symbol_model(num_classes=3).to(DEVICE)



if SYMBOL_CKPT:

    ckpt  = torch.load(SYMBOL_CKPT, map_location='cpu')

    state = ckpt.get('model_state_dict', ckpt)

    symbol_model.load_state_dict({k.replace('module.', ''): v for k, v in state.items()})

    print(f'✓ 符号检测权重已加载：{SYMBOL_CKPT}')

else:

    print('⚠ 使用随机权重（仅验证流程），请训练后传入 SYMBOL_CKPT')

### 4.1 符号检测推理

In [ ]:
def simple_nms(boxes, scores, threshold):

    """纯 numpy NMS，合并跨 tile 的重复检测框"""

    if len(boxes) == 0:

        return np.array([], dtype=np.int64)

    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]

    areas = (x2-x1+1) * (y2-y1+1)

    order = scores.argsort()[::-1]

    keep  = []

    while order.size > 0:

        i = order[0]; keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])

        yy1 = np.maximum(y1[i], y1[order[1:]])

        xx2 = np.minimum(x2[i], x2[order[1:]])

        yy2 = np.minimum(y2[i], y2[order[1:]])

        iou = (np.maximum(0,xx2-xx1+1)*np.maximum(0,yy2-yy1+1)) / \

              (areas[i] + areas[order[1:]] - np.maximum(0,xx2-xx1+1)*np.maximum(0,yy2-yy1+1))

        order = order[np.where(iou <= threshold)[0] + 1]

    return np.array(keep, dtype=np.int64)





def run_symbol_detection(model, image):

    """滑动窗口符号检测，返回 boxes/labels/scores（原图坐标系）"""

    model.eval()

    orig_h, orig_w  = image.shape[:2]

    all_boxes, all_labels, all_scores = [], [], []



    with torch.no_grad():

        for tile_img, tx, ty in extract_tiles(image):

            # Faster-RCNN 内部自带归一化，只需 ToTensor

            tensor = transforms.ToTensor()(tile_img).to(DEVICE)

            preds  = model([tensor])[0]

            if len(preds['boxes']) == 0:

                continue

            boxes = preds['boxes'].cpu().numpy()

            # 还原到原图坐标系

            boxes[:, [0,2]] = (boxes[:, [0,2]] + tx).clip(0, orig_w)

            boxes[:, [1,3]] = (boxes[:, [1,3]] + ty).clip(0, orig_h)

            all_boxes.append(boxes)

            all_labels.append(preds['labels'].cpu().numpy())

            all_scores.append(preds['scores'].cpu().numpy())



    if not all_boxes:

        return np.empty((0,4)), np.empty(0,dtype=int), np.empty(0)



    all_boxes  = np.concatenate(all_boxes)

    all_labels = np.concatenate(all_labels)

    all_scores = np.concatenate(all_scores)

    keep       = simple_nms(all_boxes, all_scores, NMS_THRESH)

    return all_boxes[keep], all_labels[keep], all_scores[keep]





t0 = time.perf_counter()

boxes, labels, scores = run_symbol_detection(symbol_model, image)

print(f'耗时：{time.perf_counter()-t0:.2f}s')

print(f'门：{(labels==1).sum()} 个  |  窗：{(labels==2).sum()} 个')

## 5. 最终叠加可视化

In [ ]:
overlay = image.copy().astype(np.float32)

overlay[wall_mask, 0] = overlay[wall_mask, 0] * 0.4 + 255 * 0.6   # 墙体染红

overlay[wall_mask, 1] = overlay[wall_mask, 1] * 0.4

overlay[wall_mask, 2] = overlay[wall_mask, 2] * 0.4

overlay = overlay.clip(0, 255).astype(np.uint8)



color_map = {1: (0, 0, 255), 2: (0, 200, 0)}

cls_name  = {1: 'door', 2: 'window'}

for box, label, score in zip(boxes, labels, scores):

    color = color_map.get(int(label), (128,128,128))

    x1,y1,x2,y2 = box.astype(int)

    cv2.rectangle(overlay, (x1,y1), (x2,y2), color, 2)

    cv2.putText(overlay, f"{cls_name.get(int(label),'?')} {score:.2f}",

                (x1, max(y1-5,0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)



fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(image)

axes[0].set_title('原始平面图', fontsize=14)

axes[0].axis('off')

axes[1].imshow(overlay)

axes[1].set_title('分析结果（红=墙体，蓝=门，绿=窗）', fontsize=14)

axes[1].axis('off')



legend = [

    mpatches.Patch(color='red',   label='wall'),

    mpatches.Patch(color='blue',  label='door'),

    mpatches.Patch(color='green', label='window'),

]

axes[1].legend(handles=legend, loc='lower right', fontsize=12)

plt.tight_layout()

plt.show()

## 6. 保存结果

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)

out.mkdir(parents=True, exist_ok=True)

stem = Path(IMAGE_PATH).stem



# 墙体概率热力图

prob_color = cv2.applyColorMap((wall_prob*255).astype(np.uint8), cv2.COLORMAP_HOT)

cv2.imwrite(str(out/f'{stem}_wall_prob.png'), prob_color)



# 墙体二值 mask

cv2.imwrite(str(out/f'{stem}_wall_mask.png'), (wall_mask*255).astype(np.uint8))



# 叠加图

cv2.imwrite(str(out/f'{stem}_overlay.png'), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))



print(f'✓ 结果已保存至：{out}/')

print(f'  {stem}_wall_prob.png')

print(f'  {stem}_wall_mask.png')

print(f'  {stem}_overlay.png')

## 7. 下一步


| 阶段 | 内容 |

|---|---|

| **训练** | 用 CubiCasa5K 数据集训练两个模型，替换随机权重 |

| **第二步** | 从墙体 mask 提取矢量多边形（论文 Section 2.4）|

| **第三步** | 生成 3D 模型并导出 IFC（论文 Section 2.5）|

| **重构** | 流程跑通后拆分成 `config.py` / `preprocessing.py` / `wall_segmentation.py` 等模块 |